# Pulling Weather data

In [5]:
## This script doesn't work yet because I already pulled data - can follow the pulling_weather_datareadme.txt
# Building this via API call is probably best and more robust if we want to keep this data set up to data/current

In [7]:
import requests
import pandas as pd
import os
from io import StringIO

# Load station list
# referene the station mapping file that already has filtered to just Ontario stations
stations_df = pd.read_csv(
    '/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/output_data/fsa_central_weather_station_mapping.csv'
)

# Filter Ontario stations with valid Station IDs
ontario_stations = stations_df[
    (stations_df['Station ID'].notna())
]

# remove duplicate stations
ontario_stations = ontario_stations.drop_duplicates(subset=['Station ID', 'Name'])


# Output folder for combined files
output_dir = "/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/output_data/weather_hourly_combined"
os.makedirs(output_dir, exist_ok=True)

# Base URL for hourly data
base_url = "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"

# Loop through each station
for _, row in ontario_stations.iterrows():
    station_id = int(row['Station ID'])
    station_name = str(row['Name']).replace(" ", "_").replace("/", "-")
    print(f"\n📥 Downloading full data for {station_name} ({station_id})...")

    station_df_list = []

    for year in [2023, 2024]:
        for month in range(1, 13):
            params = {
                "format": "csv",
                "stationID": station_id,
                "Year": year,
                "Month": month,
                "Day": 1,
                "timeframe": 1,
                "submit": "Download+Data"
            }

            response = requests.get(base_url, params=params)

            if response.status_code == 200 and "No Data Found" not in response.text:
                try:
                    # No metadata – just read the CSV directly
                    monthly_df = pd.read_csv(StringIO(response.text))
                    station_df_list.append(monthly_df)
                except Exception as e:
                    print(f"⚠️ Error reading data for {station_name} {year}-{month:02d}: {e}")
            else:
                print(f"⚠️ No data for {station_name} ({station_id}) - {year}-{month:02d}")

    if station_df_list:
        combined_df = pd.concat(station_df_list, ignore_index=True)
        combined_filename = f"{station_name}_{station_id}_2023_2024_combined.csv"
        combined_df.to_csv(os.path.join(output_dir, combined_filename), index=False)
        print(f"✅ Saved combined file: {combined_filename}")
    else:
        print(f"🚫 No data to save for {station_name} ({station_id})")



📥 Downloading full data for OTTAWA_MACDONALD-CARTIER_INT'L_A (4337)...
✅ Saved combined file: OTTAWA_MACDONALD-CARTIER_INT'L_A_4337_2023_2024_combined.csv

📥 Downloading full data for ALFRED (42006)...
✅ Saved combined file: ALFRED_42006_2023_2024_combined.csv

📥 Downloading full data for MOOSE_CREEK_WELLS (41738)...
✅ Saved combined file: MOOSE_CREEK_WELLS_41738_2023_2024_combined.csv

📥 Downloading full data for BROCKVILLE_CLIMATE (47567)...
✅ Saved combined file: BROCKVILLE_CLIMATE_47567_2023_2024_combined.csv

📥 Downloading full data for RIDEAU_NARROWS_LOCKS (26861)...
✅ Saved combined file: RIDEAU_NARROWS_LOCKS_26861_2023_2024_combined.csv

📥 Downloading full data for KILLALOE (4293)...
✅ Saved combined file: KILLALOE_4293_2023_2024_combined.csv

📥 Downloading full data for BELLEVILLE_QUINTE (55238)...
✅ Saved combined file: BELLEVILLE_QUINTE_55238_2023_2024_combined.csv

📥 Downloading full data for BANCROFT_AUTO (26799)...
✅ Saved combined file: BANCROFT_AUTO_26799_2023_2024_com